In [1]:
# Load packages
import os.path as op
import glob
import shutil
import mne
import numpy as np
import pandas as pd

In [2]:
# Prepare helper functions
def create_header_row(original_row):
    """
    Create a header row based on the first row of a trial
    """
    header_row = original_row.copy()
    header_row['eeg_trigger'] = 99
    header_row['t_stim'] = 0
    header_row['sequence'] = None
    header_row['position'] = None

    return header_row

def create_extended_df(data_frame):
    """
    Create a new df with an extra header row
    """
    new_rows = []
    for i in range(0, len(data_frame), 4):
        # Create header row from first row of trial
        header_row = create_header_row(data_frame.iloc[i])
        new_rows.append(pd.DataFrame([header_row], index=[i - 0.5]))
        
        # Add original trial rows
        new_rows.append(data_frame.iloc[i:i+4])

    return pd.concat(new_rows).sort_index().reset_index(drop=True)

def align_df_with_array(df, modified_array, pattern_column='eeg_trigger'):
    """
    Align DataFrame with a modified array by removing rows that were dropped from the complete array
    """
    # Convert DataFrame column to numpy array for comparison
    complete_array = df[pattern_column].values
    
    # Find which indices from complete_array are missing in modified_array
    complete_indices = []
    modified_indices = []
    
    i, j = 0, 0
    while i < len(complete_array) and j < len(modified_array):
        if complete_array[i] == modified_array[j]:
            complete_indices.append(i)
            modified_indices.append(j)
            i += 1
            j += 1
        else:
            # This value was dropped from complete_array
            i += 1
    # Now remove rows from DataFrame that correspond to dropped indices
    rows_to_keep = complete_indices  # These indices survived in the modified array
  
    return df.iloc[rows_to_keep].reset_index(drop=True)

In [5]:
# Define data paths
root_dir = 'C:/Users/mvmigem/Documents/data/project_2/'
raw_data_dir = op.join(root_dir,"raw/")
# get individual paths
sub_paths = glob.glob(raw_data_dir + "sub*")
# Destination path
destinantion_path = op.join(root_dir, "aligned_behav/")

In [6]:
# Remove block triggers
values_keep = [99,11,12,13,14,21,22,23,24,31,32,33,34,41,42,43,44]
# List to check if it fits everywhere
check_fits = []

for i, path in enumerate(sub_paths):

    subject = i + 1
    # Read EEG part
    raw_path = op.join(sub_paths[i],'eeg','*.bdf')
    raw_fname = glob.glob(raw_path) [0]
    raw = mne.io.read_raw_bdf(raw_fname, preload = False)
    # Get and adjust event structure
    events = mne.find_events(raw, shortest_event=1)
    events = events[np.isin(events[:, -1], values_keep)]
    # Behaviour part
    behav_path = op.join(sub_paths[i],'behav','*.csv')
    behav_fname = glob.glob(behav_path) [0]
    behav = pd.read_csv(behav_fname)
    # Add header row
    extended_df = create_extended_df(behav)
    # Align data
    aligned_df = align_df_with_array(extended_df, events[:,2])

    if all(events[:,2] == aligned_df['eeg_trigger'].to_numpy()):
        aligned_df.to_csv(destinantion_path + f"aligned-behaviour-sub-{subject:02}.csv")


Extracting EDF parameters from C:\Users\mvmigem\Documents\data\project_2\raw\sub-01\eeg\main_01.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
4530 events found on stim channel Status
Event IDs: [   11    13    22    24    31    33    42    44    99 65536 65635 65789]
Extracting EDF parameters from C:\Users\mvmigem\Documents\data\project_2\raw\sub-02\eeg\main_02.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
4526 events found on stim channel Status
Event IDs: [   11    13    22    24    31    33    42    44    99 65536 65635 65789]
Extracting EDF parameters from C:\Users\mvmigem\Documents\data\project_2\raw\sub-03\eeg\